# Rule-based FAQ chatbot
This section demonstrates the simplest possible chatbot logic: matching user questions to predefined answers using keywords.

In [1]:
# We start with a simple FAQ dictionary.
#
# A dictionary stores information as key-value pairs:
# - the key is a keyword that we expect to appear in a customer question
# - the value is the answer that the chatbot should return
#
# This approach is very transparent and easy to understand.
# However, it only works well when the user uses words that we have explicitly included as keys.

faq = {
    "refund": "Refunds are processed within 7 business days.",
    "delivery": "Standard delivery takes 2-4 business days.",
    "invoice": "Invoices are sent by email after payment confirmation.",
    "premium": "Premium customers receive priority support within 2 hours.",
    "cancel": "Orders can be cancelled within 24 hours of purchase."
}

In [2]:
# This is an example customer question.
#
# The question contains the word "premium".
# Because "premium" is one of the keys in the FAQ dictionary,
# the rule-based chatbot should be able to find the correct answer.

question = "I am a premium customer. How fast will support answer?"

In [3]:
# We create a Boolean variable to remember whether the chatbot found an answer.
#
# At the beginning, no answer has been found yet, so the value is False.

answer_found = False

# The loop goes through every keyword-answer pair in the FAQ dictionary.
#
# Example:
# keyword = "refund"
# answer = "Refunds are processed within 7 business days."

for keyword, answer in faq.items():

    # We convert the customer question to lowercase.
    #
    # This makes matching more robust:
    # "Premium", "premium", and "PREMIUM" will be treated the same way.

    if keyword in question.lower():

        # If the keyword is found in the question,
        # the chatbot prints the corresponding answer.

        print(answer)

        # We update the variable to show that an answer was found.

        answer_found = True

# If no keyword was found, the chatbot prints a generic fallback message.
#
# This fallback is important because otherwise the chatbot would simply return nothing.

if not answer_found:
    print("I do not have enough information. Please contact customer support.")

Premium customers receive priority support within 2 hours.


In [4]:
# Now we test the rule-based chatbot on several different customer questions.
#
# This helps us demonstrate both:
# - cases where keyword matching works
# - cases where keyword matching fails

test_questions = [
    "How long does delivery take?",        # matches because it contains "delivery"
    "Can I get a refund?",                 # matches because it contains "refund"
    "When will my package arrive?"         # no match because "package" is not an FAQ key
]

# We process every test question one by one.

for question in test_questions:

    # Again, we start by assuming that no answer has been found.

    matched = False

    # We compare the question with every keyword in the FAQ dictionary.

    for key, answer in faq.items():

        # If the keyword appears in the question,
        # we print the question and the matching answer.

        if key in question.lower():
            print(f"{question}")
            print(f"{answer}\n")

            # We mark the question as successfully matched.

            matched = True

            # We stop checking other keywords,
            # because we already found a relevant answer.

            break

    # If no keyword matched, we show that the rule-based approach failed.

    if not matched:
        print(f"{question}")
        print("No answer found - keyword matching failed!\n")

How long does delivery take?
Standard delivery takes 2-4 business days.

Can I get a refund?
Refunds are processed within 7 business days.

When will my package arrive?
No answer found - keyword matching failed!



In [5]:
# This is exactly why we need embeddings-based search.
#
# The question "When will my package arrive?" is related to delivery,
# but it does not use the word "delivery".
#
# A keyword-based chatbot cannot understand this semantic connection.
# An embeddings-based chatbot can compare meanings, not only words.

# Better chatbot with embeddings
This section demonstrates a more flexible chatbot that searches by meaning rather than exact keywords.

In [6]:
# We import the tools needed for semantic search.
#
# SentenceTransformer:
# - converts text into embeddings
# - embeddings are numerical representations of meaning
#
# cosine_similarity:
# - compares embeddings
# - higher similarity means that two texts are closer in meaning
#
# pandas:
# - useful for displaying and organizing results

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [7]:
# The knowledge base contains full answers that the chatbot can choose from.
#
# Unlike the earlier FAQ dictionary, this knowledge base does not rely on short keywords.
# Each entry is a complete sentence representing a possible answer.

knowledge_base = [
    "Refunds are processed within 7 business days.",
    "Standard delivery takes 2-4 business days.",
    "Invoices are sent by email after payment confirmation.",
    "Premium customers receive priority support within 2 hours.",
    "Orders can be cancelled within 24 hours of purchase."
]

# We load a pre-trained sentence embedding model.
#
# all-MiniLM-L6-v2 is a compact model that creates embeddings for sentences.
# It is often used for semantic search and similarity comparison.
#
# The model has already been trained, so we do not train it ourselves here.

model = SentenceTransformer("all-MiniLM-L6-v2")

# We convert every knowledge base answer into an embedding.
#
# After this step, each answer is represented as a vector of numbers.
# These vectors allow the computer to compare the meanings of answers.

kb_embeddings = model.encode(knowledge_base)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# This is a new customer question.
#
# Notice that it includes the words "package" and "broken",
# but our knowledge base does not contain a perfect answer about damaged packages.
#
# This is useful for demonstrating that semantic search is not magic:
# it will find the closest available answer, even if the knowledge base is incomplete.

customer_question = "My package arrived broken. What should I do?"

# We convert the customer question into an embedding,
# using the same model that we used for the knowledge base.

question_embedding = model.encode([customer_question])

# We compare the question embedding with all knowledge base embeddings.
#
# The result is a list of similarity scores:
# - one score for each knowledge base entry
# - higher score means more similar meaning

scores = cosine_similarity(question_embedding, kb_embeddings)[0]

In [9]:
# We print the original customer question.

print(f"Customer question: {customer_question}\n")

# We print the similarity score for each knowledge base entry.
#
# This makes the model's retrieval process easier to understand:
# participants can see why a particular answer was selected.

print("Similarity to each knowledge base entry:\n")

for i, (entry, score) in enumerate(zip(knowledge_base, scores)):

    # This creates a simple text-based bar.
    #
    # The longer the bar, the higher the similarity score.
    # It is only for visualization in the notebook output.

    bar = "█" * int(score * 20)

    # We print:
    # - the number of the knowledge base entry
    # - the similarity score rounded to two decimals
    # - the visual bar
    # - the answer text

    print(f"  [{i+1}] {score:.2f}  {bar}")
    print(f"       {entry}")

# argmax() returns the index of the highest similarity score.
#
# This is the entry that the embedding-based chatbot considers the best match.

best_match_index = scores.argmax()

# We print the best matching answer and its score.

print(f"\nBest match [{best_match_index+1}]: {knowledge_base[best_match_index]}")
print(f" Similarity score: {round(scores[best_match_index], 2)}")

Customer question: My package arrived broken. What should I do?

Similarity to each knowledge base entry:

  [1] 0.14  ██
       Refunds are processed within 7 business days.
  [2] 0.27  █████
       Standard delivery takes 2-4 business days.
  [3] 0.15  ██
       Invoices are sent by email after payment confirmation.
  [4] 0.19  ███
       Premium customers receive priority support within 2 hours.
  [5] 0.11  ██
       Orders can be cancelled within 24 hours of purchase.

Best match [2]: Standard delivery takes 2-4 business days.
 Similarity score: 0.27000001072883606


In [ ]:
# Important interpretation:
#
# The model selects the answer with the highest semantic similarity.
# However, if the knowledge base does not contain a truly appropriate answer,
# the model may still return the "least bad" option.
#
# This is why real chatbots need:
# - a good knowledge base
# - confidence thresholds
# - fallback responses
# - human escalation for unclear cases

# Another example: comparing keyword matching and embedding search
This section compares two chatbot approaches side by side.

In [10]:
# We define a function for a rule-based business chatbot.
#
# A function allows us to reuse the same keyword-matching logic
# for many different customer questions.

def business_chatbot(question):

    # Convert the question to lowercase to make keyword matching case-insensitive.

    question = question.lower()

    # The chatbot checks manually defined conditions in sequence.
    #
    # If the question contains "damaged" or "broken",
    # it returns an answer about damaged products.

    if "damaged" in question or "broken" in question:
        return "Please report the damaged product using the complaint form."

    # If the question contains "refund",
    # it returns the refund-related answer.

    elif "refund" in question:
        return "Refunds are processed within 7 business days."

    # If the question contains "premium",
    # it returns the premium-support answer.

    elif "premium" in question:
        return "Premium customers receive priority support within 2 hours."

    # If the question contains "invoice",
    # it returns the invoice-related answer.

    elif "invoice" in question:
        return "Invoices are sent by email after payment confirmation."

    # If none of the above keywords appear,
    # the chatbot uses a generic fallback message.

    else:
        return "I do not have enough information to answer this question."


# These are test questions used to compare both approaches.
#
# Some questions contain exact keywords.
# One question uses different wording and intentionally does not match the keyword rules.

questions = [
    "My product arrived broken. What should I do?",
    "When will I receive my refund?",
    "I am a premium customer. How fast is support?",
    "Where can I download my invoice?",
    "When will my package arrive?"   # intentional no-match for keyword bot; semantically related to delivery
]

In [11]:
# We compare the rule-based chatbot with the embedding-based search.
#
# For each customer question:
# 1. The keyword chatbot tries to answer using manually defined rules.
# 2. The embedding approach converts the question into a vector.
# 3. The embedding approach compares the question with all knowledge base answers.
# 4. The most similar answer is selected.

print("Keyword-based chatbot vs. embedding search\n")

for q in questions:

    # First, get the answer from the rule-based chatbot.

    keyword_answer = business_chatbot(q)

    # Next, convert the current customer question into an embedding.

    q_emb = model.encode([q])

    # Compare this question embedding with all knowledge base embeddings.

    s = cosine_similarity(q_emb, kb_embeddings)[0]

    # Select the answer with the highest similarity score.

    embedding_answer = knowledge_base[s.argmax()]

    # Store the highest similarity score.
    #
    # This score helps us evaluate how confident the embedding match may be.

    match_score = round(s.max(), 2)

    # Print both answers so participants can compare them.

    print(f" {q}")
    print(f"  Keyword bot : {keyword_answer}")
    print(f"  Embedding   : {embedding_answer}  (score: {match_score})")
    print()

Keyword-based chatbot vs. embedding search

 My product arrived broken. What should I do?
  Keyword bot : Please report the damaged product using the complaint form.
  Embedding   : Premium customers receive priority support within 2 hours.  (score: 0.20000000298023224)

 When will I receive my refund?
  Keyword bot : Refunds are processed within 7 business days.
  Embedding   : Refunds are processed within 7 business days.  (score: 0.7099999785423279)

 I am a premium customer. How fast is support?
  Keyword bot : Premium customers receive priority support within 2 hours.
  Embedding   : Premium customers receive priority support within 2 hours.  (score: 0.7699999809265137)

 Where can I download my invoice?
  Keyword bot : Invoices are sent by email after payment confirmation.
  Embedding   : Invoices are sent by email after payment confirmation.  (score: 0.5799999833106995)

 When will my package arrive?
  Keyword bot : I do not have enough information to answer this question.
  Emb

# Discussion
Which approach handled better the question: **"When will my package arrive?"**

The keyword bot returns a fallback because it does not find the exact word **delivery**.  
The embedding model can connect **package arrive** with the meaning of delivery.

## TASK: Improving a Business Chatbot

Work in small groups.

Imagine you are building a chatbot for an online store.

### Step 1
Add 3 new questions and answers to the knowledge base.

### Step 2
Create 5 customer questions that use different wording than the knowledge base.

### Step 3
Test both approaches:

- Keyword-based chatbot
- Embedding-based chatbot

### Discussion

1. Which questions were handled better by the embedding-based chatbot?
2. Which questions were difficult for both approaches?
3. What are the main advantages of keyword matching?
4. What are the main advantages of embeddings?
5. Which approach would you choose for a real customer-service chatbot and why?